# 08 — SD4ft-Miner: Úloha 2
## Zkušenosti dle vzdálenosti
### 4IZ503 Projektový seminář — Ultra Marathon Running

---

### Slovní zadání

Na jakých vzdálenostech zkušenost veterána (5+ startů) nejvíce pomáhá?
Porovnáváme výkonnost závodníků různých zkušenostních kategorií
na extrémních závodech (>170 km) versus krátkých závodech (<60 km).

**Hypotéza:** Zkušenostní výhoda veterána je největší na extrémních
vzdálenostech, kde správné rozložení sil, výživa a mentální
příprava jsou kritické. Na krátkých závodech zkušenost tolik
nehraje roli — dominuje absolutní rychlost.

**Ante:** `experience_cat(subset)`, maxlen=1  
**Succ:** `speed_cat(rychlý)`  
**Frst:** `distance_cat(extremni)` — závody >170 km  
**Scnd:** `distance_cat(kratka)` — závody <60 km  

SD4ft-Miner porovnává `conf(Frst & Ante ⟹ Succ)` vs `conf(Scnd & Ante ⟹ Succ)`
— tzn. ve které zkušenostní kategorii je poměr výkonnosti extremni/kratka největší.

---

### Parametry úlohy

| Parametr | Hodnota |
|---|---|
| Procedura | SD4ft-Miner |
| FrstBase (min. počet záz. — extremni) | 100 |
| ScndBase (min. počet záz. — kratka) | 100 |
| Ratioconf (min. poměr confidence extremni/kratka) | 1.2 |
| Ante | experience_cat(subset), maxlen=1 |
| Succ | speed_cat(rychlý) |
| Frst | distance_cat(extremni) |
| Scnd | distance_cat(kratka) |
| Data | ultra_clean_cm.parquet (~6.87M záznamů) |

> ⚠️ **Metodická poznámka:** Ratioconf = conf_extremni / conf_kratka ≥ 1.2
> znamená, že daná zkušenostní skupina má alespoň o 20 % vyšší
> pravděpodobnost být v rychlé třetině na extrémních závodech
> oproti krátkým závodům.

## 1. Import a načtení dat

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from cleverminer import cleverminer
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/processed')

df_cm = pd.read_parquet(DATA_DIR / 'ultra_clean_cm.parquet')
print(f"Načteno: {len(df_cm):,} řádků")
print(f"Sloupce: {df_cm.columns.tolist()}")

## 2. Příprava dat pro úlohu

In [ ]:
# Pro tuto úlohu potřebujeme: experience_cat, distance_cat, speed_cat
cols = ['experience_cat', 'distance_cat', 'speed_cat']
df_task = df_cm[cols].dropna().copy()

print(f"Záznamy s kompletními daty: {len(df_task):,}")
print()
print("Rozložení experience_cat:")
print(df_task['experience_cat'].value_counts())
print()
print("Rozložení distance_cat:")
print(df_task['distance_cat'].value_counts())
print()
print("Rozložení speed_cat (ověření ~33/33/33):")
print((df_task['speed_cat'].value_counts() / len(df_task) * 100).round(1))
print()
# Křížová tabulka experience × distance (relevantní kategorie)
df_ed = df_task[df_task['distance_cat'].isin(['extremni', 'kratka'])]
print("Záznamy dle experience_cat × distance_cat (extremni + kratka):")
print(pd.crosstab(df_ed['experience_cat'], df_ed['distance_cat']))

## 3. CleverMiner — SD4ft-Miner úloha

In [ ]:
cm = cleverminer(df=df_task)

cm.mine(
    proc='SD4ftMiner',
    quantifiers={'FrstBase': 100, 'ScndBase': 100, 'Ratioconf': 1.2},
    ante={
        'attributes': [
            {'name': 'experience_cat', 'type': 'subset', 'minlen': 1, 'maxlen': 1}
        ],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    },
    succ={
        'attributes': [
            {'name': 'speed_cat', 'type': 'one', 'value': 'rychlý'}
        ],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    },
    frst={
        'attributes': [{'name': 'distance_cat', 'type': 'one', 'value': 'extremni'}],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    },
    scnd={
        'attributes': [{'name': 'distance_cat', 'type': 'one', 'value': 'kratka'}],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    }
)

print("\nSouhrn:")
cm.print_summary()

## 4. Výsledky

In [ ]:
print("Všechna pravidla (seřazená dle Ratioconf):")
cm.print_rulelist(sortby='ratioconf', storesorted=True)

## 5. Extrakce pravidel pro analýzu

In [ ]:
rules = []
n = cm.get_rulecount()

for i in range(1, n + 1):
    quant = cm.get_quantifiers(i)
    rule_text = cm.get_ruletext(i)

    # Parsování zkušenostní kategorie z textu pravidla
    exp_match = re.search(r'experience_cat\((\w+)\)', rule_text)

    rules.append({
        'rule_id':    i,
        'experience': exp_match.group(1) if exp_match else None,
        'frstbase':   quant.get('frstbase'),
        'scndbase':   quant.get('scndbase'),
        'frstconf':   quant.get('frstconf'),
        'scndconf':   quant.get('scndconf'),
        'ratioconf':  quant.get('ratioconf'),
        'rule_text':  rule_text,
    })

df_rules = pd.DataFrame(rules)
print(f"Extrahováno {len(df_rules)} pravidel")
print()
print(df_rules.sort_values('ratioconf', ascending=False).to_string(index=False))

## 6. Vizualizace

In [ ]:
exp_order  = ['nováček', 'zkušený', 'veterán']
exp_colors = {'nováček': '#e07b54', 'zkušený': '#5b9bd5', 'veterán': '#70ad47'}
dist_labels = {'extremni': '>170 km (extremni)', 'kratka': '<60 km (kratka)'}

# Baseline confidence z dat pro všechny experience × {extremni, kratka}
baseline_data = []
for exp in exp_order:
    for dist in ['extremni', 'kratka']:
        subset = df_task[(df_task['experience_cat'] == exp) &
                         (df_task['distance_cat'] == dist)]
        if len(subset) > 0:
            conf = (subset['speed_cat'] == 'rychlý').mean()
            baseline_data.append({'experience': exp, 'distance': dist,
                                   'conf': conf, 'n': len(subset)})

df_base = pd.DataFrame(baseline_data)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('SD4ft-Miner: Zkušenosti dle vzdálenosti\n'
             '(succ: speed_cat = rychlý, Frst=extremni vs Scnd=kratka)',
             fontsize=13, fontweight='bold')

# Graf 1 — Confidence pro extremni a kratka dle zkušenosti
exps_present = [e for e in exp_order if e in df_base['experience'].values]
x = np.arange(len(exps_present))
width = 0.35

df_ext = df_base[df_base['distance'] == 'extremni'].set_index('experience')['conf']
df_krt = df_base[df_base['distance'] == 'kratka'].set_index('experience')['conf']

bars_ext = ax1.bar(x - width/2,
                   [df_ext.get(e, np.nan) for e in exps_present],
                   width, label='>170 km (extremni)', color='#e07b54', alpha=0.85, edgecolor='white')
bars_krt = ax1.bar(x + width/2,
                   [df_krt.get(e, np.nan) for e in exps_present],
                   width, label='<60 km (kratka)', color='#5b9bd5', alpha=0.85, edgecolor='white')

ax1.axhline(0.333, color='black', linewidth=1, linestyle='--', label='Průměr (33%)')

for bars in [bars_ext, bars_krt]:
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h) and h > 0:
            ax1.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=8)

ax1.set_xlabel('Zkušenostní kategorie', fontsize=11)
ax1.set_ylabel('Confidence (podíl rychlých)', fontsize=11)
ax1.set_title('Podíl rychlých: extremni vs kratka dle zkušenosti', fontsize=11)
ax1.set_xticks(x)
ax1.set_xticklabels(exps_present, fontsize=10)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0.25, 0.50)

# Graf 2 — Ratioconf (extremni/kratka) z baseline dat
ratio_data = []
for exp in exps_present:
    ext_conf = df_ext.get(exp, np.nan)
    krt_conf = df_krt.get(exp, np.nan)
    if not np.isnan(ext_conf) and not np.isnan(krt_conf) and krt_conf > 0:
        ratio_data.append({'experience': exp, 'ratioconf': ext_conf / krt_conf})

df_ratio = pd.DataFrame(ratio_data)
colors_ratio = [exp_colors.get(e, '#999') for e in df_ratio['experience'].values]

bars2 = ax2.bar(np.arange(len(df_ratio)), df_ratio['ratioconf'],
                color=colors_ratio, alpha=0.85, edgecolor='white')
for bar in bars2:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 0.005,
             f'{h:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.axhline(1.0, color='black', linewidth=1, linestyle='--', label='Parita extremni=kratka (1.0)')
ax2.axhline(1.2, color='red',   linewidth=1, linestyle=':',  label='Práh Ratioconf=1.2')

import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=exp_colors[e], alpha=0.85, label=e) for e in exps_present]
ax2.legend(handles=patches + [
    plt.Line2D([0],[0], color='black', linestyle='--'),
    plt.Line2D([0],[0], color='red',   linestyle=':')],
    labels=exps_present + ['Parita (1.0)', 'Práh 1.2'],
    fontsize=9)

ax2.set_xlabel('Zkušenostní kategorie', fontsize=11)
ax2.set_ylabel('Ratioconf (conf_extremni / conf_kratka)', fontsize=11)
ax2.set_title('Relativní výhoda na extremni vs kratka\n'
              '(vyšší = zkušenost více pomáhá na extrémech)', fontsize=11)
ax2.set_xticks(np.arange(len(df_ratio)))
ax2.set_xticklabels(df_ratio['experience'].tolist(), fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / '08_zkusenosti_vzdalenosti.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graf uložen.")

## 7. Interpretace grafů

**Graf vlevo — Confidence: extremni vs kratka dle zkušenosti:**

Ukazuje podíl rychlých závodníků pro každou kombinaci
(zkušenost × délka závodu). Pokud se confidence veterána na
extrémních závodech blíží 1/3 (průměru), zatímco nováček je
výrazně pod průměrem, zkušenost na extrémech pomáhá.

**Graf vpravo — Ratioconf (extremni/kratka) dle zkušenosti:**

Ratioconf > 1 znamená, že daná zkušenostní skupina dosahuje
relativně lepších výsledků na extrémních závodech než na krátkých.
Veterán s nejvyšším Ratioconf potvrzuje hypotézu — zkušenost
přináší největší výhodu na ultra-long vzdálenostech.

## 8. Zajímavá pravidla

In [ ]:
print("=== ZAJÍMAVÁ PRAVIDLA ===")
print()

if len(df_rules) > 0:
    df_sorted = df_rules.sort_values('ratioconf', ascending=False)
    print("TOP 3 pravidla (nejvyšší Ratioconf — největší výhoda na extremni vs kratka):")
    for _, row in df_sorted.head(3).iterrows():
        cm.print_rule(int(row['rule_id']))
        print()
else:
    print("SD4ft-Miner nenalezl pravidla splňující Ratioconf ≥ 1.2.")
    print("Žádná zkušenostní skupina nemá o 20+ % lepší výsledky na extremni vs kratka.")
    print()
    print("Baseline Ratioconf z dat:")
    print(df_ratio.to_string(index=False))

## 9. Souhrn a business interpretace

In [ ]:
print("=" * 60)
print("SOUHRN — Úloha 2 (SD4ft): Zkušenosti dle vzdálenosti")
print("=" * 60)
print()

print(f"Celkem nalezených SD4ft pravidel (Ratioconf ≥ 1.2): {len(df_rules)}")
print()

if len(df_ratio) > 0:
    print("Ratioconf (extremni/kratka) dle zkušenostní kategorie:")
    for _, row in df_ratio.sort_values('ratioconf', ascending=False).iterrows():
        marker = " ← nejvyšší" if row['ratioconf'] == df_ratio['ratioconf'].max() else ""
        print(f"  {row['experience']:10s}: {row['ratioconf']:.3f}{marker}")
    print()

    # Porovnání veterán vs nováček
    vet = df_ratio[df_ratio['experience'] == 'veterán']
    nov = df_ratio[df_ratio['experience'] == 'nováček']
    if len(vet) > 0 and len(nov) > 0:
        diff = vet.iloc[0]['ratioconf'] - nov.iloc[0]['ratioconf']
        print(f"Rozdíl Ratioconf veterán vs nováček: {diff:+.3f}")
        if diff > 0:
            print("→ Veterán má větší relativní výhodu na extremni vs kratka ✓")
            print("→ Hypotéza POTVRZENA: zkušenost nejvíce pomáhá na ultra-long vzdálenostech")
        else:
            print("→ Hypotéza se nepotvrdila — zkušenost nepomáhá více na extremni")

print()
print("BUSINESS DOPORUČENÍ:")
print("  → Tréninkový program pro první 100mi+ závod zdůraznit zkušenosti z kratších závodů")
print("  → Veterány motivovat k účasti na extremni závodech — mají největší relativní výhodu")
print("  → Race directors: kategorie 'first-timer' na kratka jsou dostupnější pro nováčky")

## Shrnutí

**Metoda:** SD4ft-Miner (CleverMiner 1.2.6). Porovnává pravidla
`experience_cat(X) ⟹ speed_cat(rychlý)` na extrémních závodech
(Frst: distance_cat=extremni) vs krátkých závodech (Scnd: distance_cat=kratka).
Pravidlo platí pokud conf_extremni / conf_kratka ≥ 1.2.

**Data:** ~6.87M závodníků, speed_cat per event.

**Klíčový nález:** Analýza Ratioconf dle zkušenostní skupiny ukazuje,
u které skupiny se zkušenost na extrémních závodech nejvíce projevuje
oproti krátkým závodům.

**Limitace:**
- speed_cat per event — srovnáváme relativní výkonnost, ne absolutní časy
- Extrémních závodů je výrazně méně než krátkých — menší statistická základna
- Definice `veterán` v datasetu nemusí přesně odpovídat 5+ startům
- SD4ft porovnává jen extremni vs kratka — chybí stredni, dlouha, casovy

**Konec série notebooků:** Pro celkový přehled viz `03_4ft_uloha1.ipynb`
a předchozí notebooky 04–07.